# 内存管理与性能分析

学习目标：能分析对象持有关系，并通过真实内存采样、计时与 CPU profile 比较优化。

前置知识：对象引用、闭包、Map、函数、数组处理和 Node.js 运行参数。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/29-memory-performance/。

1. [retention.mjs](scripts/29-memory-performance/retention.mjs)：闭包持有与显式缓存清理。
2. [weak.mjs](scripts/29-memory-performance/weak.mjs)：解引用、登记与撤销的确定性部分。
3. [memory.mjs](scripts/29-memory-performance/memory.mjs)：三个生命周期位置的实际内存采样。
4. [algorithms.mjs](scripts/29-memory-performance/algorithms.mjs)：两种等价分组实现。
5. [benchmark.mjs](scripts/29-memory-performance/benchmark.mjs)：等价断言、预热与重复计时。
6. [profile-check.mjs](scripts/29-memory-performance/profile-check.mjs)：实际采集、解析热点和删除 profile。
7. [profile-summary.mjs](scripts/29-memory-performance/profile-summary.mjs)：按节点 ID 统计并保留源码位置。
8. [profile-summary.test.mjs](scripts/29-memory-performance/profile-summary.test.mjs)：固定重名节点、不同调用路径及无效 ID 的断言。

Step 1：观察闭包持有。

```bash
node scripts/29-memory-performance/retention.mjs
```

Step 2：运行弱引用接口。

```bash
node scripts/29-memory-performance/weak.mjs
```

Step 3：运行真实内存采样。

```bash
node --expose-gc scripts/29-memory-performance/memory.mjs
```

Step 4：运行七轮基准比较。

```bash
node scripts/29-memory-performance/benchmark.mjs
```

Step 5：运行固定采样数据的统计测试。

```bash
node scripts/29-memory-performance/profile-summary.test.mjs
```

Step 6：采集并读取 CPU profile。

```bash
node scripts/29-memory-performance/profile-check.mjs
```

## 1 可达性、闭包和缓存持有

垃圾回收（garbage collection，GC）处理不再需要的内存；应用要先分析对象是否仍可从可达引用访问。闭包捕获的值、Map 中的键和值、监听器及缓存都可能让对象持续可达。把某个局部变量设为 null，只断开那一个引用，不会自动删除其他持有者。

V8 使用分代等实现策略，回收发生的时机、停顿及内存是否立即归还操作系统不由 ECMAScript 保证。下面用一个缓存保存读取器闭包：即使最初的数组变量被置空，仍能经由闭包访问数组；clear 后断开的是缓存持有关系，不能据此宣称某个精确时刻已释放全部字节。

配套 [retention.mjs](scripts/29-memory-performance/retention.mjs)：

```javascript
import assert from "node:assert/strict";
function createReader(rows) {
  return () => rows.length;
}
let rows = Array.from({ length: 1000 }, (_, index) => ({ index }));
const cache = new Map();
cache.set("report", createReader(rows));
rows = null;
assert.equal(cache.get("report")(), 1000);
console.log("reachable through closure", cache.get("report")()); // → reachable through closure 1000
cache.clear();
assert.equal(cache.size, 0);
console.log("cache references removed"); // → cache references removed；不承诺 GC 完成时刻
```

## 2 WeakRef 与 FinalizationRegistry

WeakRef 不因自身存在而强制保留目标，deref 返回仍存在的目标或 undefined。首次取得目标后，应保存在局部变量中完成当前操作；不要让业务正确性依赖它在下一次异步恢复时仍存在。当前执行期间成功解引用的对象有规范的存活一致性保障。

FinalizationRegistry 登记目标、清理回调需要的 heldValue 和可选撤销 token；目标不可达后，宿主可能安排清理回调，但不保证及时调用，也不保证进程退出前一定调用。heldValue 被登记记录强引用，不能直接或间接持有目标，否则可能阻止所期待的回收。

适用目标按 ECMAScript 2025 包括对象及可弱持有的 Symbol，不包含 Symbol.for 创建的注册 Symbol。WeakRef 缓存还可能留下大量“键 → 空弱引用”的条目，仍需容量上限或清扫策略。文件关闭、保存数据、解锁等关键逻辑不能放在最终清理回调里。

配套 [weak.mjs](scripts/29-memory-performance/weak.mjs)：

```javascript
import assert from "node:assert/strict";
let target = { name: "preview" };
const reference = new WeakRef(target);
const token = {};
const cleanupMessages = [];
const registry = new FinalizationRegistry((heldValue) => cleanupMessages.push(heldValue));
registry.register(target, "preview entry", token);
assert.equal(reference.deref(), target);
assert.equal(registry.unregister(token), true);
assert.equal(registry.unregister(token), false);
target = null;
const current = reference.deref();
assert.equal(current.name, "preview");
assert.equal(cleanupMessages.length, 0);
assert.throws(() => new WeakRef(Symbol.for("registered")), TypeError);
console.log("deref same job and unregister checked"); // → deref same job and unregister checked
// 本例不等待最终清理回调；已撤销登记，不把 GC 调度当成成功条件。
```

## 3 真实内存采样与解释边界

process.memoryUsage 返回字节计数，是 Node.js 的诊断接口。heapUsed/heapTotal 属于 V8 堆，rss 是进程驻留内存，external 包括与 JavaScript 对象关联的部分原生内存，arrayBuffers 反映缓冲区相关分配且包含在 external 中；这些数值不能不加区分地相加。

本例在分配前、缓存持有期间、清空缓存后分别采样。--expose-gc 暴露 V8 诊断扩展，Node.js 24.11.0 将此参数标为实验性；它方便观察，不是可移植的语言 API。即使请求 GC，数值也受堆容量、优化和运行时开销影响，因此不写“必须下降若干字节”的断言。

持续增长需要在相同工作负载的多轮观测中判断，再结合持有路径分析。堆快照可用于调查具体保留对象，但获取快照会暂停主线程并增加内存压力；本课只做小规模采样，不在不受控服务上直接抓取大快照。

配套 [memory.mjs](scripts/29-memory-performance/memory.mjs)：

```javascript
import assert from "node:assert/strict";
assert.equal(typeof globalThis.gc, "function", "run with --expose-gc");
function sample(label) {
  globalThis.gc();
  const { heapUsed, heapTotal, rss, external, arrayBuffers } = process.memoryUsage();
  console.log(JSON.stringify({ label, heapUsed, heapTotal, rss, external, arrayBuffers }));
}
function createReader() {
  const records = Array.from({ length: 50000 }, (_, index) => ({ index, name: `record-${index}` }));
  return () => records.length;
}
sample("before");
const cache = new Map([["report", createReader()]]);
assert.equal(cache.get("report")(), 50000);
sample("retained");
cache.clear();
assert.equal(cache.size, 0);
sample("released references");
console.log("memory samples complete"); // → memory samples complete；三次内存数字随运行变化
```

## 4 比较优化前后的正确性与耗时

优化前先固定语义。下面两个函数都对分类编号和非负小整数金额的记录求分组总额；分类编号是从 0 开始的非负整数，输入由本例生成。scanGroups 对每个分类重新扫描数组，onePass 只遍历一次并累计到 Map。本例对正常、空输入和重复分类进行结果对照，再测量相同工作负载。

performance.now 返回高精度毫秒时间戳。测试先预热，再交替改变两种实现的测量顺序，重复 7 轮，记录每轮耗时和中位数；中位数是排序后的中间值。计时不包含 console 输出，结果始终被使用并核对。

scanGroups 的重复扫描次数会随分类数增加，这是本例算法结构的直接分析；Map 具体常数成本、JIT 优化和测量比值属于引擎与机器观察。不能把一个固定“快多少倍”推广到其他数据分布，也不能接受为速度改变输出语义。

配套 [algorithms.mjs](scripts/29-memory-performance/algorithms.mjs)：

```javascript
export function scanGroups(rows) {
  const keys = [...new Set(rows.map((row) => row.group))];
  return keys.map((group) => [group,
    rows.filter((row) => row.group === group).reduce((sum, row) => sum + row.amount, 0),
  ]).sort(([left], [right]) => left - right);
}
export function onePass(rows) {
  const groups = new Map();
  for (const row of rows) groups.set(row.group, (groups.get(row.group) ?? 0) + row.amount);
  return [...groups].sort(([left], [right]) => left - right);
}
```

配套 [benchmark.mjs](scripts/29-memory-performance/benchmark.mjs)：

```javascript
import assert from "node:assert/strict";
import { performance } from "node:perf_hooks";
import { scanGroups, onePass } from "./algorithms.mjs";

for (const rows of [[], [{ group: 0, amount: 0 }], [{ group: 1, amount: 3 }, { group: 1, amount: 4 }]]) {
  assert.deepEqual(scanGroups(rows), onePass(rows));
}
const rows = Array.from({ length: 20000 }, (_, index) => ({ group: index % 100, amount: index % 17 }));
const expected = onePass(rows);
assert.deepEqual(scanGroups(rows), expected);
for (let round = 0; round < 4; round += 1) { scanGroups(rows); onePass(rows); }
const samples = { scanGroups: [], onePass: [] };
performance.mark("comparison-start");
for (let round = 0; round < 7; round += 1) {
  const functions = round % 2 === 0 ? [scanGroups, onePass] : [onePass, scanGroups];
  for (const run of functions) {
    const start = performance.now();
    const result = run(rows);
    const elapsed = performance.now() - start;
    assert.deepEqual(result, expected);
    samples[run.name].push(elapsed);
  }
}
performance.mark("comparison-end");
performance.measure("comparison", "comparison-start", "comparison-end");
function median(values) { return values.toSorted((left, right) => left - right)[3]; }
const scanMs = median(samples.scanGroups);
const passMs = median(samples.onePass);
console.log(JSON.stringify({ samplesMs: samples, medianMs: { scanGroups: scanMs, onePass: passMs }, ratio: scanMs / passMs }));
console.log("batch duration ms", performance.getEntriesByName("comparison")[0].duration);
performance.clearMarks();
performance.clearMeasures();
console.log("equivalent results; 7 trials complete"); // → equivalent results; 7 trials complete；耗时与比值为本次测量
```

## 5 采集并读取真实 CPU profile

计时回答整体花了多久，CPU profile 通过采样调用栈帮助寻找时间集中在哪里。Node.js --cpu-prof 在启动时采样并在退出前写入 .cpuprofile；--cpu-prof-interval 单位为微秒。本例使用 1000 微秒间隔，不把每一个样本解释为一次精确函数调用。

profile 的 nodes 描述调用树，children 保存子节点 ID，samples 保存每次采样的栈顶节点 ID。同一函数沿不同调用路径出现时可以有不同节点，即使 functionName 和源码位置相同也应保留各自 ID。下面按节点统计栈顶命中次数，不把子节点样本累加到父节点；函数名仅用于显示，不能作为归并键。

输出同时保留 scriptId、url、lineNumber 和 columnNumber。行列沿用协议的从 0 开始编号，表示函数位置，不代表每次采样正在执行的精确语句；无名函数显示为 (anonymous)。运行时节点可能没有普通源码 URL，负行列值按原值保留，不当作可跳转的代码位置。

配套 [profile-summary.mjs](scripts/29-memory-performance/profile-summary.mjs)：

```javascript
import assert from "node:assert/strict";

export function summarizeSamples(profile) {
  const nodes = new Map(profile.nodes.map((node) => [node.id, node]));
  const counts = new Map();
  for (const id of profile.samples) {
    assert.ok(nodes.has(id), "sample must reference a known node");
    counts.set(id, (counts.get(id) ?? 0) + 1);
  }
  return [...counts].map(([id, samples]) => {
    const frame = nodes.get(id).callFrame;
    return {
      id,
      functionName: frame.functionName || "(anonymous)",
      scriptId: frame.scriptId,
      url: frame.url,
      lineNumber: frame.lineNumber,
      columnNumber: frame.columnNumber,
      samples,
    };
  }).sort((left, right) => right.samples - left.samples || left.id - right.id);
}
```

固定测试中，4 和 5 是同一函数在 callerA、callerB 两条路径下的节点，6 是另一文件中的同名函数，7 和 8 是不同位置的匿名函数。应保留五行，样本数分别为 2、1、1、1、1；未知节点 ID 必须报错。这些数据只用于统计测试，不代表真实性能测量。

配套 [profile-summary.test.mjs](scripts/29-memory-performance/profile-summary.test.mjs)：

```javascript
import assert from "node:assert/strict";
import { summarizeSamples } from "./profile-summary.mjs";

// 固定测试数据，只保留统计所需字段和用于说明调用路径的 children。
const frame = (functionName, lineNumber) => ({
  functionName, scriptId: "1", url: "file:///fixture-a.mjs", lineNumber, columnNumber: 0,
});
const profile = {
  nodes: [
    { id: 1, callFrame: frame("root", 0), children: [2, 3] },
    { id: 2, callFrame: frame("callerA", 1), children: [4, 6, 7] },
    { id: 3, callFrame: frame("callerB", 2), children: [5, 8] },
    { id: 4, callFrame: frame("work", 10) },
    { id: 5, callFrame: frame("work", 10) },
    { id: 6, callFrame: { ...frame("work", 10), scriptId: "2", url: "file:///fixture-b.mjs" } },
    { id: 7, callFrame: frame("", 20) },
    { id: 8, callFrame: frame("", 30) },
  ],
  samples: [4, 4, 5, 6, 7, 8],
};
const rows = summarizeSamples(profile);
assert.deepEqual(rows.map(({ id, samples }) => [id, samples]), [[4, 2], [5, 1], [6, 1], [7, 1], [8, 1]]);
assert.equal(rows.filter((row) => row.functionName === "work").length, 3);
assert.equal(rows.filter((row) => row.functionName === "(anonymous)").length, 2);
assert.deepEqual(rows[0], {
  id: 4, functionName: "work", scriptId: "1", url: "file:///fixture-a.mjs",
  lineNumber: 10, columnNumber: 0, samples: 2,
});
assert.equal(rows[2].url, "file:///fixture-b.mjs");
console.log("same names and distinct paths kept separate"); // → same names and distinct paths kept separate
assert.throws(() => summarizeSamples({ ...profile, samples: [99] }), {
  name: "AssertionError", message: "sample must reference a known node",
});
console.log("unknown sample rejected"); // → unknown sample rejected
```

profile-check 在章节目录下创建专用临时子目录，启动独立被测进程，等待退出后读回真实 profile，调用上述统计函数并显示前五个节点。断言全部样本都有对应节点、统计总数与 samples 长度一致，并确认子进程完成正确性检查；随后删除这次 profile。

采样包含模块加载、GC 和其他运行时开销，函数内联也会影响栈形状。先据节点及位置定位候选热点，再用不带 profiler 的基准比较，避免把测量工具的额外成本当作优化收益。

配套 [profile-check.mjs](scripts/29-memory-performance/profile-check.mjs)：

```javascript
import { rm } from "node:fs/promises";
import assert from "node:assert/strict";
import { execFileSync } from "node:child_process";
import { mkdtempSync, readFileSync } from "node:fs";
import { resolve, relative, isAbsolute } from "node:path";
import { summarizeSamples } from "./profile-summary.mjs";
const root = import.meta.dirname;
const directory = mkdtempSync(resolve(root, "js-c-profile-"));
try {
  const output = execFileSync(process.execPath, [
    "--cpu-prof", "--cpu-prof-interval=1000", `--cpu-prof-dir=${directory}`,
    "--cpu-prof-name=sample.cpuprofile", resolve(root, "benchmark.mjs"),
  ], { encoding: "utf8", windowsHide: true });
  assert.match(output, /equivalent results; 7 trials complete/);
  const profile = JSON.parse(readFileSync(resolve(directory, "sample.cpuprofile"), "utf8"));
  assert.ok(profile.samples.length > 0);
  const rows = summarizeSamples(profile);
  assert.equal(rows.reduce((sum, row) => sum + row.samples, 0), profile.samples.length);
  const top = rows.slice(0, 5);
  console.log("sampled nodes", JSON.stringify(top)); // → 节点 ID、函数名、源码位置及样本数随本次采样变化
  console.log("CPU profile parsed"); // → CPU profile parsed
} finally {
  const child = relative(root, directory);
  assert.ok(child.startsWith("js-c-profile-") && !isAbsolute(child) && !child.includes(".."));
  await rm(directory, { recursive: true });
}
console.log("profile cleaned"); // → profile cleaned
```

## 本章小结

- 内存问题先分析仍可达的引用，GC 与最终清理回调都不承诺精确时机。
- 测量先保证输出等价，再区分内存指标、整体耗时与采样热点。
- 优化收益只对已测工作负载成立，诊断工具的开销需要单独考虑。

## 练习

1. 不清空缓存再做第二轮分配；标准：能说明两批数据为何被持有，并对照实际 heapUsed，不能只凭一个数值下结论。
2. 把分类数从 100 改为 5 和 500；标准：两实现仍输出相同结果，各重复 7 轮后比较中位数与比值。
3. 在保持结果一致的前提下减少一次临时数组分配；标准：先过等价断言，再测量与读取 profile，记录收益是否稳定。

## 参考与引用来源

- TC39（ECMA-262 第 16 版）：[§26 WeakRef 与 FinalizationRegistry](https://tc39.es/ecma262/2025/multipage/managing-memory.html)，deref、register、unregister；[§9.10 WeakRef 执行关系](https://tc39.es/ecma262/2025/multipage/executable-code-and-execution-contexts.html#sec-weakref-processing-model)：可弱持有目标与存活一致性。
- GitHub / TC39：[WeakRefs 提案说明](https://github.com/tc39/proposal-weakrefs#a-note-of-caution)，A note of caution、Weak references、Finalizers：缓存持有、回调不确定性；具体目标类型以第 16 版为准。
- Node.js：24.11.0 [fsPromises.rm](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisesrmpath-options)：临时目录清理；[process.memoryUsage](https://nodejs.org/download/release/v24.11.0/docs/api/process.html#processmemoryusage)、[performance.now/mark/measure](https://nodejs.org/download/release/v24.11.0/docs/api/perf_hooks.html)、[--cpu-prof 与 --expose-gc](https://nodejs.org/download/release/v24.11.0/docs/api/cli.html#--cpu-prof)；[内存管理实现与指标](https://nodejs.org/en/learn/diagnostics/memory/understanding-and-tuning-memory)、[堆快照的成本](https://nodejs.org/en/learn/diagnostics/memory/using-heap-snapshot)：诊断条件及限制。
- Chrome DevTools Protocol：[Profiler.Profile 的 samples](https://chromedevtools.github.io/devtools-protocol/v8/Profiler/#type-Profile)、[Profiler.ProfileNode 的 id、callFrame 和 children](https://chromedevtools.github.io/devtools-protocol/v8/Profiler/#type-ProfileNode)、[Runtime.CallFrame 的函数名、脚本及行列位置](https://chromedevtools.github.io/devtools-protocol/v8/Runtime/#type-CallFrame)：CPU profile 节点身份、调用树与定位字段。